### import

In [1]:
import sys
sys.path.append('./')
from utils import CharmBulkInfo
charm_info = CharmBulkInfo.CharmBulk()
bulk_info = CharmBulkInfo.Bulk()

### config

In [2]:
file_path = '/home/wuct/MetaData/DATA/PbPb/2023/pass4/charmbulk/flook/AnalysisResults.root'
pt_bins = [0.2, 1, 2, 3, 4, 5, 6]


#### THn QA

In [ ]:
import ROOT
colors = [
    # ROOT.kBlack,       
    ROOT.kRed-4,       
    ROOT.kBlue-4, 
    ROOT.kGreen+2,     
    ROOT.kOrange+7,   
    # ROOT.kYellow-7,    
    ROOT.kMagenta-3,   
    ROOT.kCyan-3,      
    ROOT.kSpring-5,    
    ROOT.kViolet-4,    
    ROOT.kTeal-5,    
    ROOT.kGray+1    
]
file = ROOT.TFile(file_path, 'READ')
thn_charm = file.Get(charm_info.thurl)
thn_bulk = file.Get(bulk_info.thurl)

pt_mins = pt_bins[:-1]
pt_maxs = pt_bins[1:]

temp_thn_charm = thn_charm.Clone('temp_thn_charm')
temp_thn_charm.GetAxis(charm_info.axis_id('eta')).SetRangeUser(-0.8, -0.2)
a_side_charm = temp_thn_charm.Clone('a_side_charm')
temp_thn_bulk = thn_bulk.Clone('temp_thn_bulk')
a_side_bulk = temp_thn_bulk.Clone('a_side_bulk')

temp_thn_charm = thn_charm.Clone('temp_thn_charm')
temp_thn_charm.GetAxis(charm_info.axis_id('eta')).SetRangeUser(0.2, 0.8)
b_side_charm = temp_thn_charm.Clone('b_side_charm')
temp_thn_bulk = thn_bulk.Clone('temp_thn_bulk')
b_side_bulk = temp_thn_bulk.Clone('b_side_bulk')

hMeanPt_bs = []
hMeanPt_as = []
hNum_bs = []
hNum_as = []
hPtProduct_bs = []
hPtProduct_as = []
for i_pt, (pt_min, pt_max) in enumerate(zip(pt_mins, pt_maxs)):
    print(f'pt bin {i_pt}: {pt_min} - {pt_max} GeV/c')
    # a side
    ## track mean pt for each pt of charm
    a_side_charm.GetAxis(charm_info.axis_id('pT')).SetRangeUser(pt_min, pt_max)
    hMeanPt_as.append(a_side_charm.Projection(charm_info.axis_id('mean_pt_b'))) # opposite side for mean pt
    
    ## track number of tracks for each pt of charm
    hNum_as.append(a_side_bulk.Projection(bulk_info.axis_id('ntrk_b'))) # opposite side for number of tracks
    
    ## pt product for each pt of charm
    hPtProduct_as.append(a_side_charm.Projection(charm_info.axis_id('pt_product'))) # slice by eta for pt product
    
    # b side
    ## track mean pt for each pt of charm
    b_side_charm.GetAxis(charm_info.axis_id('pT')).SetRangeUser(pt_min, pt_max)
    hMeanPt_bs.append(b_side_charm.Projection(charm_info.axis_id('mean_pt_a'))) # opposite side for mean pt
    
    ## track number of tracks for each pt of charm
    hNum_bs.append(b_side_bulk.Projection(bulk_info.axis_id('ntrk_a'))) # opposite side for number of tracks
    
    ## pt product for each pt of charm
    hPtProduct_bs.append(b_side_charm.Projection(charm_info.axis_id('pt_product'))) # slice by eta for pt product

output = ROOT.TFile('charm_bulk_results.root', 'RECREATE')

# canvas for mean pt in different charm pt bins
# a side
a_max_y_mean_pt = max(h.GetMaximum() for h in hMeanPt_as)
cMeanPt_as = ROOT.TCanvas('cMeanPt_as', 'cMeanPt_as', 1600, 1200)
leg_as = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_as.SetHeader('a side charm: -0.8 < #eta < -0.2')
leg_as.SetTextSize(0.03)
meanLines = []
for i_pt, hMeanPt_a in enumerate(hMeanPt_as):
    hMeanPt_a.SetLineColor(colors[i_pt])
    hMeanPt_a.SetLineWidth(3)
    hMeanPt_a.SetMarkerColor(colors[i_pt])
    hMeanPt_a.SetMarkerSize(4)
    meanLine = ROOT.TLine(hMeanPt_a.GetMean(), 0, hMeanPt_a.GetMean(), a_max_y_mean_pt)
    meanLines.append(meanLine)
    meanLine.SetLineColor(colors[i_pt])
    meanLine.SetLineStyle(ROOT.kDashed)
    meanLine.SetLineWidth(3)
    # hMeanPt_a.SetTitle(f'Mean track pT in different charm pT bins (a side charm)')
    if i_pt == 0:
        hMeanPt_a.Draw('l')
        hMeanPt_a.GetYaxis().SetRangeUser(0, a_max_y_mean_pt * 1.2)
    else:
        hMeanPt_a.Draw('l same')
    meanLine.Draw('same')

    leg_as.AddEntry(hMeanPt_a, f'{pt_mins[i_pt]} < pT < {pt_maxs[i_pt]} GeV/c', 'lp')
# cMeanPt_as.BuildLegend()
leg_as.Draw()
output.cd()
cMeanPt_as.Write()
cMeanPt_as.SaveAs('cMeanPt_as.png')

# b side
b_max_y_mean_pt = max(h.GetMaximum() for h in hMeanPt_bs)
cMeanPt_bs = ROOT.TCanvas('cMeanPt_bs', 'cMeanPt_bs', 1600, 1200)
leg_bs = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_bs.SetHeader('b side charm: 0.2 < #eta < 0.8')
leg_bs.SetTextSize(0.03)
meanLines = []
for i_pt, hMeanPt_b in enumerate(hMeanPt_bs):
    hMeanPt_b.SetLineColor(colors[i_pt])
    hMeanPt_b.SetMarkerColor(colors[i_pt])
    hMeanPt_b.SetLineWidth(3)
    hMeanPt_b.SetMarkerSize(4)
    meanLines.append(ROOT.TLine(hMeanPt_b.GetMean(), 0, hMeanPt_b.GetMean(), b_max_y_mean_pt))
    meanLine = meanLines[-1]
    meanLine.SetLineColor(colors[i_pt])
    meanLine.SetLineStyle(ROOT.kDashed)
    meanLine.SetLineWidth(3)
    # hMeanPt_b.SetTitle(f'Mean track pT in different charm pT bins (b side charm)')
    if i_pt == 0:
        hMeanPt_b.Draw('l')
        hMeanPt_b.GetYaxis().SetRangeUser(0, b_max_y_mean_pt * 1.2)
    else:
        hMeanPt_b.Draw('l same')
    meanLine.Draw('same')
    leg_bs.AddEntry(hMeanPt_b, f'{pt_mins[i_pt]} < pT < {pt_maxs[i_pt]} GeV/c', 'lp')
# cMeanPt_bs.BuildLegend()
leg_bs.Draw()
output.cd()
cMeanPt_bs.Write()
cMeanPt_bs.SaveAs('cMeanPt_bs.png')
ou

# canvas for number of tracks in different charm pt bins
# a side
cNum_as = ROOT.TCanvas('cNum_as', 'cNum_as', 1600, 1200)
leg_num_as = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_num_as.SetHeader('a side charm: -0.8 < #eta < -0.2')
leg_num_as.SetTextSize(0.03)
for i_pt, hNum_a in enumerate(hNum_as):
    if i_pt == 0:
        hNum_a.GetXaxis().SetRangeUser(0, 400)
    else:
        continue
    hNum_a.SetLineColor(i_pt + 1)
    hNum_a.SetLineWidth(2)
    hNum_a.SetMarkerSize(4)
    # hNum_a.SetTitle(f'Number of tracks in different charm pT bins (a side charm)')
    hNum_a.Draw('same')
    leg_num_as.AddEntry(hNum_a, '30-40%', 'lp')
# cNum_as.BuildLegend()
leg_num_as.Draw()
cNum_as.SaveAs('cNum_as.png')
# b side
cNum_bs = ROOT.TCanvas('cNum_bs', 'cNum_bs', 1600, 1200)
leg_num_bs = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_num_bs.SetHeader('b side charm: 0.2 < #eta < 0.8')
leg_num_bs.SetTextSize(0.03)
for i_pt, hNum_b in enumerate(hNum_bs):
    if i_pt == 0:
        hNum_b.GetXaxis().SetRangeUser(0, 400)
    else:
        continue
    hNum_b.SetLineColor(i_pt + 1)
    hNum_b.SetLineWidth(2)
    hNum_b.SetMarkerSize(4)
    # hNum_b.SetTitle(f'Number of tracks in different charm pT bins (b side charm)')
    hNum_b.Draw('same')
    leg_num_bs.AddEntry(hNum_b, '30-40%', 'lp')
# cNum_bs.BuildLegend()
leg_num_bs.Draw()
cNum_bs.SaveAs('cNum_bs.png')

# canvas for pt product in different charm pt bins
# a side
a_max_y_pt_product = max(h.GetMaximum() for h in hPtProduct_as)
a_min_y_pt_product = min(h.GetMinimum() for h in hPtProduct_as)
cPtProduct_as = ROOT.TCanvas('cPtProduct_as', 'cPtProduct_as', 1600, 1200)
leg_pt_product_as = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_pt_product_as.SetHeader('a side charm: -0.8 < #eta < -0.2')
leg_pt_product_as.SetTextSize(0.03)
for i_pt, hPtProduct_a in enumerate(hPtProduct_as):
    if i_pt == 0:
        hPtProduct_a.GetXaxis().SetRangeUser(0, 10)
        hPtProduct_a.GetYaxis().SetRangeUser(0, a_max_y_pt_product * 1.2)
    hPtProduct_a.SetLineColor(colors[i_pt])
    hPtProduct_a.SetLineWidth(3)
    hPtProduct_a.SetMarkerColor(colors[i_pt])
    hPtProduct_a.SetMarkerSize(4)
    # hPtProduct_a.SetTitle(f'Pt product in different charm pT bins (a side charm)')
    if i_pt == 0:
        hPtProduct_a.Draw('l')
    else:
        hPtProduct_a.Draw('l same')
    leg_pt_product_as.AddEntry(hPtProduct_a, f'{pt_mins[i_pt]} < pT < {pt_maxs[i_pt]} GeV/c', 'lp')
#cPtProduct_as.BuildLegend()
leg_pt_product_as.Draw()
cPtProduct_as.SaveAs('cPtProduct_as.png')
# b side
b_max_y_pt_product = max(h.GetMaximum() for h in hPtProduct_bs)
b_min_y_pt_product = min(h.GetMinimum() for h in hPtProduct_bs)
cPtProduct_bs = ROOT.TCanvas('cPtProduct_bs', 'cPtProduct_bs', 1600, 1200)
leg_pt_product_bs = ROOT.TLegend(0.6, 0.6, 0.9, 0.9)
leg_pt_product_bs.SetHeader('b side charm: 0.2 < #eta < 0.8')
leg_pt_product_bs.SetTextSize(0.03)
for i_pt, hPtProduct_b in enumerate(hPtProduct_bs):
    if i_pt == 0:
        hPtProduct_b.GetXaxis().SetRangeUser(0, 10)
        hPtProduct_b.GetYaxis().SetRangeUser(0, b_max_y_pt_product * 1.2)
    hPtProduct_b.SetLineColor(colors[i_pt])
    hPtProduct_b.SetLineWidth(3)
    hPtProduct_b.SetMarkerColor(colors[i_pt])
    hPtProduct_b.SetMarkerSize(4)
    # hPtProduct_b.SetTitle(f'Pt product in different charm pT bins (b side charm)')
    if i_pt == 0:
        hPtProduct_b.Draw('l')
    else:
        hPtProduct_b.Draw('l same')
    leg_pt_product_bs.AddEntry(hPtProduct_b, f'{pt_mins[i_pt]} < pT < {pt_maxs[i_pt]} GeV/c', 'lp')
# cPtProduct_bs.BuildLegend()
leg_pt_product_bs.Draw()
cPtProduct_bs.SaveAs('cPtProduct_bs.png')


AttributeError: 'TStyle' object has no attribute 'SetDirectory'